# Урок 17 — Поліморфізм, Інкапсуляція та Дандер-методи

**Модуль 2 · Python Intermediate · Автор: Viktor Nikoriak**

---

## Про що цей урок?

У попередньому уроці ми з'ясували, що наслідування — це **пошук методів через MRO**, а не копіювання коду.  
Але справжня архітектура живе в іншому просторі: в тому, **як об'єкти взаємодіють між собою** та **як вони захищають свій внутрішній стан**.

> **Три головні ідеї уроку:**
> 1. **Поліморфізм** — один інтерфейс, різна поведінка (і чому спадкування для цього не потрібне)
> 2. **Інкапсуляція** — захист інваріантів об'єкта від зовнішнього хаосу
> 3. **Дандер-методи** — як ваші класи інтегруються у Python Runtime

---

## Зміст уроку

| # | Тема | Ключова ідея |
|---|------|--------------|
| 1 | Поліморфізм: вбудований та через наслідування | Один виклик → різна поведінка |
| 2 | Качина типізація (Duck Typing) | Python перевіряє поведінку, а не тип |
| 3 | Інкапсуляція та інваріанти | Об'єкт захищає свій стан |
| 4 | `@property`: Pythonic getter/setter | Валідація без синтаксичного шуму |
| 5 | Дандер-методи: `__init__`, `__str__`, `__repr__` | Інтеграція з Python Runtime |
| 6 | Перевантаження операторів: `__add__`, `__len__` | Ваш клас поводиться як вбудований тип |
| 7 | Викликувані об'єкти: `__call__` | Функція з пам'яттю |

---

## Частина 1: Поліморфізм — «Один виклик, різна поведінка»

Поліморфізм (від грецьк. *«багато форм»*) — здатність одного й того самого виклику метода  
запускати **різну логіку** залежно від типу об'єкта.

Ви використовуєте поліморфізм з першого дня роботи з Python — навіть не підозрюючи про це.

### Передбачення

Що виведе функція `double_value(x)` для різних типів аргументів?

In [ ]:
# Поліморфізм оператора + — він вже вбудований у Python!
def double_value(x):
    return x + x  # один і той самий оператор, різна поведінка

print(double_value(5))          # математичне додавання
print(double_value("abc"))      # конкатенація рядків
print(double_value([1, 2, 3]))  # конкатенація списків

**Пояснення:**

Функція `double_value` не знає, який тип вона отримає.  
Вона просто використовує оператор `+`. Python сам вирішує, що робити — залежно від типу об'єкта.

Це і є суть поліморфізму: **один інтерфейс → різна реалізація**.

---

### Класичний поліморфізм через наслідування

У класичному ООП поліморфізм реалізується через успадкування та **перевизначення методів (overriding)**.  
Батьківський клас задає **інтерфейс**, дочірні класи — **специфічну поведінку**.

### Передбачення

Ми будуємо медіаплеєр. Клас `AudioFile` є базовим. Як Python вирішить, який метод `play()` викликати у циклі?

In [ ]:
# Базовий клас визначає ІНТЕРФЕЙС (контракт)
class AudioFile:
    def __init__(self, filename):
        self.filename = filename

    def play(self):
        # Базова реалізація — просто нагадування про контракт
        raise NotImplementedError("Підклас повинен реалізувати play()")


# Дочірні класи надають специфічну РЕАЛІЗАЦІЮ
class MP3File(AudioFile):
    def play(self):
        print(f"Граємо {self.filename} як MP3. Розпакування audio...")


class OggFile(AudioFile):
    def play(self):
        print(f"Граємо {self.filename} як OGG. Декодування vorbis потоку...")


# Медіаплеєр — жодного if/elif з перевіркою типів!
playlist = [MP3File("пісня.mp3"), OggFile("подкаст.ogg"), MP3File("ремікс.mp3")]

for track in playlist:
    track.play()  # Python сам викликає правильну версію методу

**Пояснення:**

Медіаплеєр не перевіряє тип об'єкта. Він просто викликає `.play()`.  
Сам об'єкт знає, **як саме** він має відтворюватися.

Це позбавляє нас від крихкого коду типу:
```python
# ПОГАНО — так не треба:
if isinstance(track, MP3File):
    play_mp3(track)
elif isinstance(track, OggFile):
    play_ogg(track)
```

Якщо завтра розробник додасть `FlacFile` — медіаплеєр запрацює з ним **без жодної зміни в коді**.

---

## Частина 2: Качина типізація — Python не перевіряє паспорт

Класичний поліморфізм вимагає **спільного предка** (наслідування).  
Але Python пішов далі. Він застосовує принцип **Duck Typing** (качина типізація):

> *«Якщо воно ходить як качка і крякає як качка — це качка»*

Python не перевіряє, **чим є** об'єкт. Він перевіряє, **що він вміє робити**.

### Передбачення

Клас `FlacFile` **не** успадковує `AudioFile`. Що станеться, якщо додати його до плейлисту?

In [ ]:
# FlacFile — повністю незалежний клас, без наслідування від AudioFile!
class FlacFile:
    def __init__(self, filename):
        self.filename = filename

    def play(self):
        print(f"Граємо {self.filename} як FLAC. Lossless аудіо без втрат!")


# Наш медіаплеєр — нічого не знає про FlacFile
def play_all(tracks):
    for track in tracks:
        track.play()  # Python просто шукає метод .play() на об'єкті


# Змішуємо об'єкти з різних ієрархій — і все працює!
playlist = [
    MP3File("пісня.mp3"),
    FlacFile("симфонія.flac"),  # FlacFile — не нащадок AudioFile!
    OggFile("подкаст.ogg"),
]

play_all(playlist)

**Пояснення:**

В Python немає формальної перевірки «ти справжній AudioFile?» перед викликом `.play()`.  
Якщо метод існує — Python його виконає.

Це дає нам **величезну гнучкість**:

| Мова | Як перевіряється поліморфізм |
|------|------------------------------|
| Java / C++ | Під час компіляції: потрібен спільний тип або інтерфейс |
| **Python** | Під час виконання: достатньо мати потрібний метод |

> **Архітектурний наслідок:**  
> Якщо розробник винайде `VinylRecord` через 5 років і додасть до нього `.play()` —  
> наш `play_all` запрацює з ним **без жодного рядка змін**.

### Бізнес-приклад: система нарахування зарплат

### Передбачення

У циклі є `WellPaidDuck` — качка. Чи може вона бути в одному списку з працівниками?

In [ ]:
# Різні типи "платіжних одиниць" — без спільного предка
class CommissionEmployee:
    def earnings(self):
        return 1500.00

class SalariedEmployee:
    def earnings(self):
        return 5000.00

class WellPaidDuck:
    # Качка — явно не є працівником, але теж вміє рахувати earnings()
    def earnings(self):
        return 1_000_000.00


# Система нарахування зарплат: не перевіряє тип, тільки поведінку
payroll = [CommissionEmployee(), SalariedEmployee(), WellPaidDuck()]

total = 0
for entity in payroll:
    amount = entity.earnings()
    print(f"Виплачуємо: ${amount:,.2f}")
    total += amount

print(f"\nЗагальний фонд: ${total:,.2f}")

**Пояснення:**

Качка не є працівником — але вона реалізує той самий **неформальний інтерфейс** (метод `earnings()`).  
Python обробляє її так само, як і звичайних працівників.

> **Найкращий вид поліморфізму** — ненавмисний: ви пишете функцію, а потім виявляєте,  
> що вона вже працює з об'єктами, про які ви навіть не думали.

---

### Завдання: Duck Typing у дії

Створіть клас `Hacker` з методом `reply()`, що виводить `"Hello World!"`.  
Створіть клас `Cat` з методом `reply()`, що виводить `"Нявк"`.  
Напишіть функцію `communicate(entity)`, яка викликає `.reply()` на будь-якому об'єкті.

### Аналогія з реального світу: кермо автомобіля

Уявіть, що ви сідаєте за кермо автомобіля.  
Ваш **інтерфейс** — це кермо і педаль газу. Ви не знаєте і не думаєте про те,  
чи під капотом V8-двигун внутрішнього згоряння, чи електробатарея Tesla.

Ви натискаєте педаль (виклик методу) — автомобіль вирішує, як саме прискоритись (реалізація).

```
Водій              →  pedal.press()   → інтерфейс
CombustionEngine   →  inject fuel     → реалізація 1
ElectricEngine     →  discharge cells → реалізація 2
```

**Це і є поліморфізм в архітектурі:**  
Клієнтський код (водій) залежить від **інтерфейсу**, а не від **конкретної реалізації**.  
Коли заводи перейдуть на водневі двигуни — водію не потрібно буде переучуватися.

In [ ]:
# Ваш код тут:


class Hacker:
    pass  # замініть pass на метод reply()

class Cat:
    pass  # замініть pass на метод reply()


def communicate(entity):
    pass  # ваша реалізація тут


# Після реалізації розкоментуйте:
# communicate(Hacker())
# communicate(Cat())

---

### Дебагінг: коли Duck Typing ламається

Duck Typing — двосічний меч. Він гнучкий, але помилки виявляються лише під час виконання.  

**Завдання на дебагінг:**  
Функція `checkout` очікує об'єкт із методом `get_total()`. Чому код падає і як виправити `Basket`?

In [ ]:
def checkout(cart_like_object):
    # Функція не перевіряє тип — тільки поведінку
    print(f"Ваш рахунок: ${cart_like_object.get_total():.2f}")


class ShoppingCart:
    def get_total(self):
        return 45.99


class Basket:
    def __init__(self):
        self.total_price = 12.50  # є ДАНІ, але нема ПОВЕДІНКИ!


checkout(ShoppingCart())  # Працює
checkout(Basket())        # ПАДАЄ! AttributeError: 'Basket' object has no attribute 'get_total'

# Підказка: Basket має потрібні дані (total_price),
# але не реалізує потрібний ІНТЕРФЕЙС (метод get_total())

---

## Частина 2b: Темний бік наслідування — чому воно ламає системи

Спадкування часто подають як «найкращий спосіб повторного використання коду».  
Але в архітектурі програмного забезпечення воно визнається  
**найжорсткішою формою залежності**, яку можна ввести в кодову базу.

### Проблема 1: Крихкий базовий клас (Fragile Base Class Problem)

Будь-яка зміна у **батьківському класі** миттєво поширюється вниз по ієрархії.  
Невинна зміна на вершині може зламати підкласи, яких ніхто не чіпав.

**Приклад з транспортними засобами:**  
Розробник додав `change_spark_plug()` до базового класу `Vehicle`.  
Тепер `LunarRover` — місячний всюдихід на електробатареях — успадковує метод  
«заміни свічки», якого у нього немає і бути не може.

In [ ]:
# Демонстрація: ефект хвилі (ripple effect)

class Vehicle:
    def __init__(self, name):
        self.name = name

    def start(self):
        print(f"{self.name}: запускаємось!")

    # Розробник додав новий метод "безпечно" для Car і Motorcycle
    def change_spark_plug(self):
        print(f"{self.name}: замінюємо свічку запалення...")


class Car(Vehicle):
    pass

class Motorcycle(Vehicle):
    pass

class LunarRover(Vehicle):
    # Місячний всюдихід НЕ МАЄ свічок — він електричний!
    # Але він успадковує цей метод через ієрархію
    pass


rover = LunarRover("Apollo 17 Rover")
rover.start()

# BUG: LunarRover успадкував метод, який йому не потрібен і не має сенсу!
rover.change_spark_plug()   # "запускаємось!" — абсурд
print(isinstance(rover, Vehicle))  # True — rover "є" транспортним засобом зі свічками

### Проблема 2: Принцип підстановки Ліскова (LSP)

> **LSP:** Об'єкт підкласу має бути **повністю замінюваним** на об'єкт батьківського класу  
> без порушення логіки програми.

**Пастка Rectangle vs Square:**  
Математично квадрат є прямокутником. Але в ООП наслідування `Square` від `Rectangle`  
це класична катастрофа — порушення LSP.

**Чому:**  
Якщо система очікує `Rectangle`, вона припускає: `set_width(10)` **тільки** змінює ширину.  
Але `Square` мусить тримати ширину = висоті. Підстановка ламає логіку.

In [ ]:
# Демонстрація порушення LSP

class Rectangle:
    def __init__(self, width, height):
        self._width = width
        self._height = height

    def set_width(self, w):
        self._width = w

    def set_height(self, h):
        self._height = h

    def area(self):
        return self._width * self._height

    def __repr__(self):
        return f"Rectangle({self._width}x{self._height})"


class Square(Rectangle):
    # Квадрат ПОРУШУЄ LSP: set_width змінює обидві сторони
    def set_width(self, w):
        self._width = w
        self._height = w   # примусова рівність сторін

    def set_height(self, h):
        self._width = h    # те саме
        self._height = h


def check_rectangle_behavior(shape: Rectangle):
    """Функція очікує Rectangle і покладається на його поведінку."""
    shape.set_width(5)
    shape.set_height(10)
    expected_area = 50  # 5 * 10
    actual_area = shape.area()
    print(f"{shape}: площа = {actual_area}, очікувалось = {expected_area}")
    assert actual_area == expected_area, f"LSP ПОРУШЕНО! {actual_area} != {expected_area}"


print("--- Rectangle ---")
check_rectangle_behavior(Rectangle(2, 3))  # Працює

print("--- Square ---")
try:
    check_rectangle_behavior(Square(5, 5))  # ЛАМАЄТЬСЯ! AssertionError
except AssertionError as e:
    print(f"Порушення LSP: {e}")
    print("Square НЕ є справжнім Rectangle в сенсі ООП")

### Проблема 3: Зловживання наслідуванням для повторного використання коду

Початківці часто наслідують лише щоб «взяти чужий код», навіть коли відношення «is-a» не існує.

**Приклад 1 — WizardCustomer:**  
Є клас `WizCoin` (монети) із методом `weight_in_grams()`.  
Розробник створює `WizardCustomer` і **наслідує** `WizCoin`, щоб отримати грошову логіку.  
Тепер `wizard.weight_in_grams()` повертає вагу монет чарівника, а не самого чарівника.  
Клієнт **не є** монетою — це хибний дизайн.

**Приклад 2 — Calculator vs Stack:**  
Потрібен калькулятор. Є клас `Stack` із `push()` і `pop()`.  
Розробник наслідує `Stack`, щоб отримати стек «безкоштовно».  
Але калькулятор **не є** стеком — і тепер `calculator.pop()` є публічним методом,  
який бачить кожен користувач класу. Це хаос.

In [ ]:
# Погані приклади (is-a без справжнього сенсу)

class WizCoin:
    """Клас, що відстежує монети чарівника."""
    def __init__(self, galleons, sickles, knuts):
        self.galleons = galleons
        self.sickles = sickles
        self.knuts = knuts

    def value(self):
        return self.galleons * 17 * 29 + self.sickles * 29 + self.knuts

    def weight_in_grams(self):
        # Вага монет (galleons = 25г, sickles = 11г, knuts = 5г)
        return self.galleons * 25 + self.sickles * 11 + self.knuts * 5


# ПОГАНО: клієнт "є" монетою лише щоб отримати грошову логіку
class WizardCustomerBad(WizCoin):
    def __init__(self, name):
        super().__init__(galleons=10, sickles=5, knuts=3)  # гаманець захардкоджено!
        self.name = name


wizard = WizardCustomerBad("Гаррі Поттер")
print(f"{wizard.name} коштує {wizard.value()} кнатів")
# Абсурд: weight_in_grams() повертає вагу МОНЕТ, а не чарівника!
print(f"Вага чарівника: {wizard.weight_in_grams()} г")  # 268 г — це вага монет!
print(f"Чи є Гаррі монетою? {isinstance(wizard, WizCoin)}")  # True — це дизайн-помилка

### Рішення: Композиція замість наслідування

> **«Надавайте перевагу композиції об'єктів над наслідуванням класів»**  
> — Банда чотирьох (Design Patterns, 1994)

Замість відношення «є різновидом» (is-a) — будуємо відношення «містить як частину» (has-a).  
Об'єкти стають **Lego-блоками**: їх можна замінювати, тестувати та комбінувати незалежно.

In [ ]:
# ДОБРЕ: Композиція замість наслідування

# --- Двигуни як окремі незалежні об'єкти ---
class CombustionEngine:
    def start(self):
        print("Двигун внутрішнього згоряння: Vroooom!")

    def change_spark_plug(self):
        print("Заміна свічки запалення...")


class ElectricEngine:
    def start(self):
        print("Електродвигун: тихий старт, повний момент!")

    # Немає change_spark_plug — електродвигун свічок не має


# --- Транспортні засоби містять двигун (has-a) ---
class Car:
    def __init__(self, name: str):
        self.name = name
        self.engine = CombustionEngine()  # has-a відношення

    def drive(self):
        self.engine.start()
        print(f"{self.name} їде дорогою!")


class LunarRoverGood:
    def __init__(self, name: str):
        self.name = name
        self.engine = ElectricEngine()  # has-a: інший двигун

    def drive(self):
        self.engine.start()
        print(f"{self.name} їде по поверхні Місяця!")


# --- WizardCustomer правильно ---
class WizardCustomerGood:
    def __init__(self, name: str, galleons: int, sickles: int, knuts: int):
        self.name = name
        self.purse = WizCoin(galleons, sickles, knuts)  # has-a: гаманець

    def buy(self, item_cost: int):
        if self.purse.value() >= item_cost:
            print(f"{self.name} купує товар за {item_cost} кнатів")
        else:
            print(f"{self.name} не має достатньо коштів!")


# Демонстрація
car = Car("BMW")
rover = LunarRoverGood("Apollo Rover")
harry = WizardCustomerGood("Гаррі", 10, 5, 3)

car.drive()
print()
rover.drive()
# rover.engine.change_spark_plug() — так, але LunarRover не "є" транспортом зі свічками

print()
harry.buy(100)
# Зміна WizCoin не зламає WizardCustomerGood!
print(f"Вага монет Гаррі: {harry.purse.weight_in_grams()} г")  # логічно зрозуміло

**Порівняння підходів:**

| Критерій | Наслідування | Композиція |
|----------|-------------|------------|
| Зв'язування | Жорстке — зміна батька ламає дітей | Слабке — компоненти ізольовані |
| Замінюваність | Складна | Легка — передаємо інший об'єкт |
| Тестування | Потрібно тестувати весь ланцюг | Компоненти тестуються окремо |
| Прозорість інтерфейсу | Всі методи батька публічні | Контролюємо, що відкривати |
| Інтуїтивність | «Є різновидом» — часто брехня | «Містить» — завжди правда |

---

## Частина 3: Інкапсуляція — захист інваріантів

**Інкапсуляція** — це захист внутрішнього стану об'єкта від зовнішнього хаосу.

### Що таке інваріант?

**Інваріант** — бізнес-правило, яке завжди має бути правдивим:
- Баланс рахунку не може бути від'ємним
- Радіус кола має бути > 0
- Вага товару у кошику не може бути від'ємною

> **Мета інкапсуляції:** гарантувати, що об'єкт ніколи не потрапить у невалідний стан.

### Баг без інкапсуляції

In [ ]:
# ПРОБЛЕМА: стан відкритий — будь-хто може зламати інваріант
class BankAccountBad:
    def __init__(self, owner, initial_balance):
        self.owner = owner
        self.balance = initial_balance  # публічний атрибут — небезпечно!

    def deposit(self, amount):
        self.balance += amount
        print(f"Поповнення: +{amount}. Баланс: {self.balance}")

    def withdraw(self, amount):
        if amount > self.balance:
            print("Недостатньо коштів!")
            return
        self.balance -= amount
        print(f"Зняття: -{amount}. Баланс: {self.balance}")


account = BankAccountBad("Аліса", 1000)
account.deposit(500)
account.withdraw(200)

# Молодший розробник обходить логіку і ламає інваріант:
account.balance = -50_000  # БАГ: від'ємний баланс в обхід правил!
print(f"\nПоточний баланс: {account.balance}")  # -50000 — інваріант зруйнований

### Конвенції іменування в Python

Python не має справжніх `private`-модифікаторів. Замість них — **конвенції іменування**:

| Префікс | Приклад | Значення |
|---------|---------|----------|
| Без префіксу | `self.balance` | Публічний — можна читати та змінювати |
| `_` (одне підкреслення) | `self._balance` | Захищений — «не чіпати ззовні», але Python не забороняє |
| `__` (два підкреслення) | `self.__balance` | Name mangling — Python перейменує в `_ClassName__balance` |

> **Важливо:** `__` захищає від **випадкового** перекриття у підкласах, а не від зловмисника.

In [ ]:
# Демонстрація name mangling
class SecretKeeper:
    def __init__(self):
        self.public = "бачать усі"
        self._protected = "будь ласка, не чіпати"
        self.__private = "Python перейменував мене!"  # стане _SecretKeeper__private

    def reveal(self):
        # Всередині класу доступ через звичайне ім'я
        print(f"Секрет зсередини: {self.__private}")


keeper = SecretKeeper()

print(keeper.public)       # Працює
print(keeper._protected)   # Працює (конвенція, не заборона)
keeper.reveal()            # Працює — доступ зсередини класу

# Пряме звернення до приватного атрибута — AttributeError!
try:
    print(keeper.__private)
except AttributeError as e:
    print(f"Помилка: {e}")

# Але через name mangling — все ще можна (це не security, а safety):
print(keeper._SecretKeeper__private)  # Працює — знаємо справжнє ім'я

# Переглянемо всі атрибути об'єкта
print("\nВсі атрибути:", [a for a in dir(keeper) if not a.startswith('__')])

---

### Реальні баги без інкапсуляції

Ось три класичні катастрофи, які трапляються в реальних системах.

#### Баг 1: Колізія імен у підкласах (Name Clashing Bug)

Клас `Dog` має внутрішній атрибут `mood`. Розробник підкласу `Beagle`  
**випадково** створює власний атрибут із тим самим ім'ям — і **знищує стан батька**.

In [ ]:
# ДЕМОНСТРАЦІЯ: Баг колізії імен

class DogWithPublicMood:
    """Клас із публічним атрибутом mood — небезпечно!"""
    def __init__(self, name: str, mood: str):
        self.name = name
        self.mood = mood   # публічний — будь-хто може перекрити!

    def describe(self):
        return f"{self.name} почувається: {self.mood}"


class Beagle(DogWithPublicMood):
    def __init__(self, name: str):
        super().__init__(name, mood="happy")
        # Розробник підкласу ВИПАДКОВО перекриває атрибут батька!
        self.mood = "завжди збуджений"  # знищує батьківський mood

    def sniff(self):
        return f"{self.name} нюхає все навколо (настрій: {self.mood})"


dog = Beagle("Снупі")
print(dog.describe())   # батьківський mood перекритий
print(dog.sniff())

print()

# ВИПРАВЛЕННЯ: використовуємо name mangling (__mood)
class DogSafe:
    def __init__(self, name: str, mood: str):
        self.name = name
        self.__mood = mood   # name mangling → _DogSafe__mood

    def describe(self):
        return f"{self.name} почувається: {self.__mood}"


class BeagleSafe(DogSafe):
    def __init__(self, name: str):
        super().__init__(name, mood="happy")
        # Тепер це ОКРЕМИЙ атрибут _BeagleSafe__mood, не _DogSafe__mood!
        self.__mood = "завжди збуджений"

    def sniff(self):
        return f"{self.name} нюхає (власний настрій: {self.__mood})"


dog2 = BeagleSafe("Снупі")
print(dog2.describe())   # батьківський __mood збережений: "happy"
print(dog2.sniff())      # власний __mood підкласу: "завжди збуджений"

# Перевіримо атрибути: два незалежні "mood"
print([a for a in dir(dog2) if 'mood' in a])

#### Баг 2: Free Truffle Bug — `LineItem` без валідації

В органічному магазині клас `LineItem` відстежує вагу і ціну товару.  
Якщо атрибути публічні — помилка чи баг може встановити від'ємну вагу або нульову ціну  
на білі трюфелі вартістю $100/кг.

In [ ]:
# ПРОБЛЕМА: LineItem без захисту

class LineItemBad:
    def __init__(self, product: str, weight_kg: float, price_per_kg: float):
        self.product = product
        self.weight_kg = weight_kg       # публічні! будь-хто може зламати
        self.price_per_kg = price_per_kg

    def subtotal(self):
        return self.weight_kg * self.price_per_kg


# Нормальне використання
truffle = LineItemBad("Білий трюфель", 0.5, 100.0)
print(f"Рахунок: ${truffle.subtotal():.2f}")  # $50.00

# Баг (або зловмисне маніпулювання): інваріанти порушені
truffle.weight_kg = -1.0    # від'ємна вага — фізично неможлива!
truffle.price_per_kg = 0.0  # безкоштовний трюфель — фінансовий збиток!
print(f"Рахунок після бага: ${truffle.subtotal():.2f}")  # $0.00 — Free Truffle Bug!

print()

# РІШЕННЯ: LineItem з @property захищає інваріанти
class LineItem:
    def __init__(self, product: str, weight_kg: float, price_per_kg: float):
        self.product = product
        self.weight_kg = weight_kg       # setter спрацює одразу!
        self.price_per_kg = price_per_kg

    @property
    def weight_kg(self) -> float:
        return self._weight_kg

    @weight_kg.setter
    def weight_kg(self, value: float):
        if value <= 0:
            raise ValueError(f"Вага має бути > 0. Отримано: {value}")
        self._weight_kg = value

    @property
    def price_per_kg(self) -> float:
        return self._price_per_kg

    @price_per_kg.setter
    def price_per_kg(self, value: float):
        if value < 0:
            raise ValueError(f"Ціна не може бути від'ємною. Отримано: {value}")
        self._price_per_kg = value

    def subtotal(self) -> float:
        return self._weight_kg * self._price_per_kg


truffle2 = LineItem("Білий трюфель", 0.5, 100.0)
print(f"Рахунок: ${truffle2.subtotal():.2f}")  # $50.00

try:
    truffle2.weight_kg = -1.0   # setter захистить!
except ValueError as e:
    print(f"Захист: {e}")

try:
    truffle2.price_per_kg = 0.0  # дозволено (знижка 100%)
    print(f"Рахунок зі знижкою: ${truffle2.subtotal():.2f}")
except ValueError as e:
    print(f"Захист: {e}")

---

## Частина 4: `@property` — Pythonic спосіб захистити стан

У Java та C++ для захисту стану пишуть громіздкі `get_balance()` та `set_balance()`.  
Python вирішує цю задачу елегантно через **декоратор `@property`**.

Він дозволяє:
- Звертатися до атрибута **як до звичайного** (`account.balance = 100`)
- Але **прозоро** запускати валідаційну логіку всередині

### Передбачення

Що станеться, якщо передати від'ємний баланс або від'ємну суму зняття?

In [ ]:
class BankAccount:
    def __init__(self, owner: str, initial_balance: float):
        self.owner = owner
        # Використовуємо setter при ініціалізації — валідація вже тут!
        self.balance = initial_balance

    @property
    def balance(self) -> float:
        """Геттер: читаємо _balance"""
        return self._balance

    @balance.setter
    def balance(self, value: float):
        """Сеттер: валідуємо перед збереженням"""
        if value < 0:
            raise ValueError(f"Баланс не може бути від'ємним. Отримано: {value}")
        self._balance = value  # зберігаємо у захищений атрибут

    def deposit(self, amount: float):
        if amount <= 0:
            raise ValueError("Сума поповнення має бути > 0")
        self.balance += amount  # викликає setter → валідація автоматична
        print(f"Поповнення: +{amount:.2f}. Баланс: {self.balance:.2f}")

    def withdraw(self, amount: float):
        if amount <= 0:
            raise ValueError("Сума зняття має бути > 0")
        if amount > self.balance:
            raise ValueError(f"Недостатньо коштів. Баланс: {self.balance:.2f}")
        self.balance -= amount  # setter перевірить результат
        print(f"Зняття: -{amount:.2f}. Баланс: {self.balance:.2f}")


# Звичайне використання — синтаксис чистий, як із публічним атрибутом
account = BankAccount("Аліса", 1000)
account.deposit(500)
account.withdraw(200)
print(f"Поточний баланс: {account.balance:.2f}")

# Спробуємо зламати інваріант
print("\n--- Спроба встановити від'ємний баланс ---")
try:
    account.balance = -50_000  # setter спрацює і підніме ValueError!
except ValueError as e:
    print(f"Захист спрацював: {e}")

print(f"Баланс залишився незмінним: {account.balance:.2f}")

**Пояснення:**

```python
account.balance = -50_000  # виглядає як звичайне присвоєння...
# ...але Python тихо викликає account.balance.setter(-50_000)
# де запускається наша валідація → ValueError!
```

Зовнішній код не знає про `_balance`. Він бачить тільки чистий інтерфейс.  
Об'єкт захищає свій інваріант — **завжди**.

> **Правило:** Якщо атрибут підпадає під бізнес-правила → огорніть його у `@property`.  
> Якщо обмежень немає — залиште звичайним публічним атрибутом (не додавайте property без причини).

---

## Частина 5: Дандер-методи — Інтеграція з Python Runtime

**Дандер-методи** (Double UNDERscore, «dunder») — це спеціальні методи виду `__name__`.  
Ви не викликаєте їх напряму. Їх **автоматично** викликає Python у відповідь на вбудований синтаксис.

| Виклик | Що Python робить «під капотом» |
|--------|-------------------------------|
| `obj + other` | `obj.__add__(other)` |
| `print(obj)` | `obj.__str__()` |
| `repr(obj)` | `obj.__repr__()` |
| `len(obj)` | `obj.__len__()` |
| `obj(arg)` | `obj.__call__(arg)` |

> **Ключова ідея:** Дандер-методи — це спосіб, яким ваш клас **підключається до Python Runtime**.  
> Реалізуйте їх — і ваші об'єкти поводитимуться так само природно, як вбудовані типи.

---

### `__init__`: ініціалізатор (ви вже знаєте)

Викликається автоматично після створення об'єкта в пам'яті. Гарантує валідний початковий стан.

---

### `__str__` vs `__repr__`: два різні читачі

Python поважає різницю між **кінцевим користувачем** та **розробником**:

| Метод | Аудиторія | Коли викликається | Мета |
|-------|-----------|-------------------|------|
| `__str__` | Кінцевий користувач | `print(obj)`, f-рядки | Зрозуміло та красиво |
| `__repr__` | Розробник | `repr(obj)`, REPL | Точно та однозначно |

> **Правило:** `__repr__` має повертати рядок, з якого можна відтворити об'єкт.  
> Якщо `__str__` не визначено — Python використає `__repr__` замість нього.

### Передбачення

Що виведе `repr(v1)`, `print(v1)` і `print(v1 + v2)`?

In [ ]:
class Vector:
    """Двовимірний вектор із повною підтримкою Python Runtime."""

    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        # Для розробника: точне представлення, з якого можна відтворити об'єкт
        return f"Vector({self.x}, {self.y})"

    def __str__(self) -> str:
        # Для користувача: красиво і зрозуміло
        return f"({self.x}, {self.y})"

    def __add__(self, other: "Vector") -> "Vector":
        # Перевантаження оператора +
        # ПРАВИЛО: повертаємо НОВИЙ об'єкт, не змінюємо self або other!
        if isinstance(other, Vector):
            return Vector(self.x + other.x, self.y + other.y)
        # NotImplemented сигналізує Python: спробуй інший спосіб
        return NotImplemented


v1 = Vector(2, 3)
v2 = Vector(4, 5)

print(repr(v1))    # Викликає __repr__ → для розробника
print(v1)          # Викликає __str__ → для користувача
print(v1 + v2)     # Викликає __add__, потім __str__ для print

# Перевіримо, що оригінальні вектори не змінились (immutable операція)
print(f"v1 залишився: {v1}")
print(f"v2 залишився: {v2}")

**Пояснення:**

```
repr(v1)   → Vector(2, 3)   ← можна вставити в код і відтворити об'єкт
print(v1)  → (2, 3)         ← читабельно для користувача
v1 + v2    → (6, 8)         ← __add__ повертає новий Vector, print викликає __str__
```

> **Чому `__add__` повертає новий об'єкт?**  
> Математичні оператори не повинні мутувати існуючі об'єкти.  
> `2 + 3` не змінює `2` — воно створює нове `5`. Так само і ваші вектори.

---

## Частина 6: `__len__` та контейнерний протокол

Якщо ваш клас моделює колекцію — реалізуйте `__len__`.  
Тоді Python дозволить використовувати `len(obj)` так само, як для списків та словників.

### Передбачення

Що виведе `len(playlist)` після додавання двох треків?

In [ ]:
class Playlist:
    """Плейліст з підтримкою вбудованих функцій Python."""

    def __init__(self, name: str):
        self.name = name
        self._tracks = []  # захищений внутрішній список

    def add(self, track: str):
        """Додати трек до плейлісту."""
        self._tracks.append(track)

    def __len__(self) -> int:
        # len(playlist) → playlist.__len__()
        return len(self._tracks)

    def __repr__(self) -> str:
        return f"Playlist('{self.name}', {len(self)} треків)"

    def __str__(self) -> str:
        if not self._tracks:
            return f"Плейліст '{self.name}' порожній"
        tracks_str = "\n".join(f"  {i+1}. {t}" for i, t in enumerate(self._tracks))
        return f"Плейліст '{self.name}':\n{tracks_str}"


my_playlist = Playlist("Ранкова підбірка")
print(f"Кількість треків: {len(my_playlist)}")  # 0

my_playlist.add("пісня.mp3")
my_playlist.add("симфонія.flac")
my_playlist.add("подкаст.ogg")

print(f"Кількість треків: {len(my_playlist)}")  # 3
print(repr(my_playlist))   # __repr__
print()
print(my_playlist)         # __str__

---

## Частина 7: `__call__` — Функція з пам'яттю

Звичайна функція «забуває» свій стан після виконання. Локальні змінні знищуються.  
Але інколи потрібна **функція, яка пам'ятає** — скільки разів її викликали, що обробляла, тощо.

Це вирішує `__call__`: якщо об'єкт має цей метод — його можна **викликати як функцію** (`obj()`).

> **Де це використовується:**
> - Кешування та мемоїзація
> - Rate limiter (обмежувач запитів)
> - Stateful декоратори
> - Складні алгоритми, розбиті на методи всередині класу

### Передбачення

Чим `api_limiter("User_Alice")` відрізняється від звичайного виклику функції?

In [ ]:
class RateLimiter:
    """Обмежувач запитів — функція, яка пам'ятає свій стан між викликами."""

    def __init__(self, max_requests: int):
        self.max_requests = max_requests
        self.requests_made = 0    # стан зберігається між викликами!
        self.history = []

    def __call__(self, user_id: str) -> str:
        # obj(user_id) → obj.__call__(user_id)
        if self.requests_made >= self.max_requests:
            return f"Заблоковано: {user_id} перевищив ліміт."

        self.requests_made += 1
        self.history.append(user_id)
        return f"Дозволено: запит {self.requests_made}/{self.max_requests} від {user_id}."

    def __repr__(self) -> str:
        return f"RateLimiter(max={self.max_requests}, used={self.requests_made})"


# Створюємо ОДИН раз — як звичайний об'єкт
api_limiter = RateLimiter(max_requests=3)

# Використовуємо ЯК ФУНКЦІЮ — але він пам'ятає стан!
print(api_limiter("User_Alice"))   # 1/3 — дозволено
print(api_limiter("User_Bob"))     # 2/3 — дозволено
print(api_limiter("User_Alice"))   # 3/3 — дозволено
print(api_limiter("User_Eve"))     # заблоковано!
print(api_limiter("User_Alice"))   # заблоковано!

print(f"\nСтан обмежувача: {repr(api_limiter)}")
print(f"Історія: {api_limiter.history}")

**Пояснення:**

```python
api_limiter("User_Alice")   # виглядає як виклик функції...
# ...але насправді: api_limiter.__call__("User_Alice")
# і між кожним викликом api_limiter.requests_made зберігається!
```

Звичайна функція не може цього зробити — вона не пам'ятає стан.  
Callable object — може. Це потужний архітектурний шаблон.

---

### `__call__` + `__len__`: CallTracker

In [ ]:
class CallTracker:
    """Відстежує кожен виклик і зберігає передані дані."""

    def __init__(self):
        self.history = []  # внутрішній стан між викликами

    def __call__(self, item):
        # Об'єкт викликається як функція: tracker(item)
        self.history.append(item)
        return f"Оброблено та збережено: {item}"

    def __len__(self):
        # len(tracker) → кількість збережених елементів
        return len(self.history)

    def __repr__(self):
        return f"CallTracker({len(self)} записів)"


tracker = CallTracker()

# Викликаємо як функцію
print(tracker("яблуко"))
print(tracker("банан"))
print(tracker("вишня"))

# Стан зберігся між викликами
print(f"\nКількість записів: {len(tracker)}")  # __len__
print(f"Об'єкт: {repr(tracker)}")             # __repr__
print(f"Історія: {tracker.history}")

---

## Частина 7b: Повний довідник дандер-методів

Дандер-методів існує кілька десятків. Нижче — систематизований огляд за категоріями  
з прикладами найважливіших із них.

### Категорія 1: Текстове подання та конвертація типів

| Метод | Тригер | Аудиторія |
|-------|--------|-----------|
| `__str__` | `print(obj)`, `str(obj)`, f-рядки | Кінцевий користувач |
| `__repr__` | `repr(obj)`, REPL | Розробник |
| `__bool__` | `bool(obj)`, `if obj:` | Будь-хто |
| `__int__` | `int(obj)` | Конвертація |
| `__float__` | `float(obj)` | Конвертація |

In [ ]:
# Демонстрація: __bool__, __int__, __float__

class Temperature:
    """Температура з повною підтримкою конвертацій Python Runtime."""

    def __init__(self, celsius: float):
        self.celsius = celsius

    def __repr__(self):
        return f"Temperature({self.celsius})"

    def __str__(self):
        return f"{self.celsius}°C"

    def __bool__(self):
        # False якщо абсолютний нуль (0 K = -273.15 C), True інакше
        return self.celsius > -273.15

    def __int__(self):
        # Конвертуємо в Кельвін (округлений)
        return int(self.celsius + 273.15)

    def __float__(self):
        return float(self.celsius)


t_room = Temperature(22.5)
t_abs_zero = Temperature(-273.15)

print(repr(t_room))          # __repr__
print(t_room)                # __str__
print(f"Температура: {t_room}")  # __str__ в f-рядку

print(f"\nbool(22.5°C) = {bool(t_room)}")          # __bool__
print(f"bool(-273.15°C) = {bool(t_abs_zero)}")    # __bool__ → False

print(f"\nint(22.5°C) = {int(t_room)} K")          # __int__ → Кельвіни
print(f"float(22.5°C) = {float(t_room)}")         # __float__

# __bool__ дозволяє використовувати у if-умовах
if t_room:
    print("\nТемпература вище абсолютного нуля.")

### Категорія 2: Математичні оператори та порівняння

**Правило для математичних методів:** завжди повертати **новий** об'єкт, не мутувати `self`.  
**Правило для in-place методів** (`__iadd__`, `__isub__`, ...): мутувати `self` і повертати його.

| Оператор | Прямий метод | Віддзеркалений | In-place |
|----------|-------------|----------------|----------|
| `a + b` | `a.__add__(b)` | `b.__radd__(a)` | `a.__iadd__(b)` → `+=` |
| `a - b` | `a.__sub__(b)` | `b.__rsub__(a)` | `a.__isub__(b)` → `-=` |
| `a * b` | `a.__mul__(b)` | `b.__rmul__(a)` | `a.__imul__(b)` → `*=` |
| `a / b` | `a.__truediv__(b)` | `b.__rtruediv__(a)` | `a.__itruediv__(b)` → `/=` |
| `a == b` | `a.__eq__(b)` | | |
| `a < b` | `a.__lt__(b)` | | |

In [ ]:
# Демонстрація: математичні оператори та порівняння на прикладі Vector

class Vector:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    # --- Математичні оператори ---
    def __add__(self, other):
        if isinstance(other, Vector):
            return Vector(self.x + other.x, self.y + other.y)
        return NotImplemented

    def __sub__(self, other):
        if isinstance(other, Vector):
            return Vector(self.x - other.x, self.y - other.y)
        return NotImplemented

    def __mul__(self, scalar):
        # Множення на скаляр: v * 3
        if isinstance(scalar, (int, float)):
            return Vector(self.x * scalar, self.y * scalar)
        return NotImplemented

    def __rmul__(self, scalar):
        # Віддзеркалений: 3 * v → викликає v.__rmul__(3)
        return self.__mul__(scalar)

    def __iadd__(self, other):
        # In-place: v += other → мутуємо self і повертаємо його
        if isinstance(other, Vector):
            self.x += other.x
            self.y += other.y
            return self
        return NotImplemented

    def __neg__(self):
        # Унарний мінус: -v
        return Vector(-self.x, -self.y)

    # --- Оператори порівняння ---
    def __eq__(self, other):
        if isinstance(other, Vector):
            return self.x == other.x and self.y == other.y
        return NotImplemented

    def __lt__(self, other):
        # Порівнюємо за довжиною вектора (корінь із суми квадратів)
        if isinstance(other, Vector):
            return (self.x**2 + self.y**2) < (other.x**2 + other.y**2)
        return NotImplemented

    def magnitude(self):
        return (self.x**2 + self.y**2) ** 0.5


v1 = Vector(2, 3)
v2 = Vector(1, 1)

print(f"v1 + v2 = {v1 + v2}")      # __add__
print(f"v1 - v2 = {v1 - v2}")      # __sub__
print(f"v1 * 3  = {v1 * 3}")       # __mul__
print(f"3 * v1  = {3 * v1}")       # __rmul__ (Python спробує int.__mul__ спочатку)
print(f"-v1     = {-v1}")          # __neg__
print(f"v1 == v2: {v1 == v2}")    # __eq__
print(f"v2 < v1: {v2 < v1}")      # __lt__ (за magnitude)

# In-place
v3 = Vector(0, 0)
v3 += v1   # __iadd__
print(f"v3 після += v1: {v3}")  # v3 мутований, не новий об'єкт

### Категорія 3: Колекції та послідовності

Ці методи дозволяють вашому класу поводитися як список або словник.

In [ ]:
# Демонстрація: контейнерні дандер-методи

class Inventory:
    """Інвентар магазину — поводиться як колекція."""
    def __init__(self):
        self._items = {}  # {name: quantity}

    def __setitem__(self, name: str, qty: int):
        # inv["яблуко"] = 50
        if qty < 0:
            raise ValueError("Кількість не може бути від'ємною")
        self._items[name] = qty

    def __getitem__(self, name: str):
        # inv["яблуко"]
        return self._items[name]  # KeyError якщо не знайдено

    def __delitem__(self, name: str):
        # del inv["яблуко"]
        del self._items[name]

    def __contains__(self, name: str):
        # "яблуко" in inv
        return name in self._items

    def __len__(self):
        # len(inv)
        return len(self._items)

    def __iter__(self):
        # for item in inv: ...
        return iter(self._items)

    def __repr__(self):
        return f"Inventory({self._items})"


inv = Inventory()

# __setitem__
inv["яблуко"] = 100
inv["банан"] = 50
inv["апельсин"] = 75
print(f"Після додавання: {repr(inv)}")

# __getitem__
print(f"\nЯблук: {inv['яблуко']}")         # 100

# __contains__
print(f"'банан' in inv: {'банан' in inv}")  # __contains__ → True
print(f"'груша' in inv: {'груша' in inv}")  # False

# __len__
print(f"\nКількість позицій: {len(inv)}")   # __len__ → 3

# __iter__
print("\nВсі товари:")
for item in inv:                             # __iter__
    print(f"  {item}: {inv[item]} шт.")

# __delitem__
del inv["банан"]
print(f"\nПісля del 'банан': {len(inv)} позиції")

### Категорія 4: Управління доступом до атрибутів

Ці методи — найнижчий рівень системи атрибутів Python.  
Вони перехоплюють звернення до `obj.attr` на рівні інтерпретатора.

| Метод | Коли викликається |
|-------|------------------|
| `__getattr__` | Атрибут **не знайдено** — останній шанс |
| `__getattribute__` | **Будь-яке** звернення до атрибута |
| `__setattr__` | Будь-яке **присвоєння** атрибута |
| `__delattr__` | Будь-яке **видалення** атрибута |

> **Обережно:** `__getattribute__` перехоплює **всі** звернення, включно з `self.x` всередині класу.  
> Якщо реалізувати його неправильно — отримаєте нескінченну рекурсію.

In [ ]:
# Демонстрація: __getattr__ та __setattr__

class StrictRecord:
    """Запис із фіксованим набором полів — забороняє довільні атрибути."""
    # Дозволений набір полів
    _ALLOWED = frozenset(['name', 'age', 'email'])

    def __setattr__(self, name: str, value):
        # Перехоплюємо БУДЬ-ЯКЕ присвоєння
        if name.startswith('_'):
            # Службові атрибути — дозволяємо
            super().__setattr__(name, value)
        elif name not in self._ALLOWED:
            raise AttributeError(
                f"Поле '{name}' не дозволено. Доступні: {self._ALLOWED}"
            )
        else:
            super().__setattr__(name, value)  # зберігаємо через батьківський метод

    def __getattr__(self, name: str):
        # Викликається ТІЛЬКИ якщо атрибут не знайдено звичайним способом
        return f"(поле '{name}' не встановлено)"


record = StrictRecord()
record.name = "Аліса"
record.age = 30
print(f"name: {record.name}")
print(f"email: {record.email}")   # __getattr__ → не встановлено

try:
    record.phone = "+380"  # __setattr__ заблокує!
except AttributeError as e:
    print(f"\nПомилка: {e}")

print()

# Практичніший приклад: логування доступу до атрибутів
class AuditedObject:
    """Об'єкт, який логує всі зміни атрибутів."""
    def __setattr__(self, name: str, value):
        if not name.startswith('_'):
            print(f"[AUDIT] {name} = {value!r}")
        super().__setattr__(name, value)


obj = AuditedObject()
obj.username = "bob"      # [AUDIT] username = 'bob'
obj.role = "admin"        # [AUDIT] role = 'admin'

### Категорія 5: Повна таблиця дандер-методів

| Категорія | Методи | Тригер |
|-----------|--------|--------|
| **Життєвий цикл** | `__init__`, `__del__` | Створення / знищення об'єкта |
| **Рядки** | `__str__`, `__repr__` | `print()`, `repr()`, f-рядки |
| **Конвертація** | `__bool__`, `__int__`, `__float__`, `__complex__` | `bool()`, `int()`, `float()` |
| **Математика** | `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__floordiv__`, `__mod__`, `__pow__`, `__matmul__` | `+`, `-`, `*`, `/`, `//`, `%`, `**`, `@` |
| **Математика (rhs)** | `__radd__`, `__rsub__`, `__rmul__`, ... | Коли лівий операнд повертає `NotImplemented` |
| **In-place** | `__iadd__`, `__isub__`, `__imul__`, ... | `+=`, `-=`, `*=` ... |
| **Унарні** | `__neg__`, `__pos__`, `__abs__` | `-x`, `+x`, `abs(x)` |
| **Порівняння** | `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__` | `==`, `!=`, `<`, `<=`, `>`, `>=` |
| **Бітові** | `__and__`, `__or__`, `__xor__`, `__lshift__`, `__rshift__`, `__invert__` | `&`, `\|`, `^`, `<<`, `>>`, `~` |
| **Контейнер** | `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__contains__`, `__iter__` | `len()`, `[]`, `in`, `for` |
| **Атрибути** | `__getattr__`, `__getattribute__`, `__setattr__`, `__delattr__` | `.attr`, `.attr = v`, `del .attr` |
| **Callable** | `__call__` | `obj()` |
| **Хешування** | `__hash__` | `hash(obj)`, використання в `dict`/`set` |
| **Контекст** | `__enter__`, `__exit__` | `with obj as ...:` |

### Бонус: `__enter__` та `__exit__` — Context Manager протокол

Це саме ті методи, які дозволяють писати `with open(...) as f:`.  
Якщо реалізувати їх у своєму класі — його можна використовувати як менеджер контексту.

In [ ]:
# Context Manager через дандер-методи

class ManagedDatabase:
    """З'єднання з БД, яке автоматично закривається."""
    def __init__(self, db_name: str):
        self.db_name = db_name
        self.connection = None

    def __enter__(self):
        # Відкрити з'єднання при вході в блок with
        print(f"[DB] Відкриваємо з'єднання з '{self.db_name}'")
        self.connection = f"conn://{self.db_name}"
        return self  # повертаємо себе → доступно як 'as db'

    def __exit__(self, exc_type, exc_val, exc_tb):
        # Закрити з'єднання при виході з блоку with (навіть при помилці!)
        print(f"[DB] Закриваємо з'єднання з '{self.db_name}'")
        self.connection = None
        # Якщо exc_type не None — була помилка
        if exc_type:
            print(f"[DB] Помилка при роботі з БД: {exc_val}")
        return False  # False = не подавляємо виняток

    def query(self, sql: str):
        if not self.connection:
            raise RuntimeError("З'єднання не відкрито!")
        return f"[Results of: {sql}]"


# Використання — автоматичне закриття гарантовано!
print("=== Нормальна робота ===")
with ManagedDatabase("production.db") as db:
    result = db.query("SELECT * FROM users")
    print(f"Результат: {result}")
# Тут з'єднання вже закрито

print()
print("=== З помилкою ===")
try:
    with ManagedDatabase("staging.db") as db:
        print(db.query("SELECT * FROM orders"))
        raise ValueError("Щось пішло не так!")  # з'єднання все одно закриється
except ValueError:
    print("Виняток передається далі")

---

## Частина 8: Патерни проєктування в Python

Класичні патерни з книги «Design Patterns» (Банда чотирьох) розроблялися  
для мов **без** функцій-першокласних об'єктів і duck typing.  
У Python багато з них можна **суттєво спростити**.

### Патерн «Стратегія» (Strategy Pattern)

**Класичне визначення:**  
Визначає сімейство алгоритмів, інкапсулює кожен із них і робить їх взаємозамінними.

**Класична реалізація** (Java-стиль):  
Абстрактний базовий клас + окремий клас для кожної стратегії з одним методом `execute()`.

**Python-спосіб:**  
Просто передайте функцію. В Python функції — це об'єкти першого класу.

In [ ]:
# Класичний підхід (Java-style) — надто багато коду для простого завдання
class SortStrategy:
    def sort(self, data):
        raise NotImplementedError

class BubbleSort(SortStrategy):
    def sort(self, data):
        return sorted(data)  # спрощено для прикладу

class QuickSort(SortStrategy):
    def sort(self, data):
        return sorted(data, reverse=False)

class DataProcessor:
    def __init__(self, strategy: SortStrategy):
        self._strategy = strategy

    def process(self, data):
        return self._strategy.sort(data)

processor = DataProcessor(BubbleSort())
print("Класичний:", processor.process([3, 1, 4, 1, 5]))

print()

# Python-спосіб — просто передаємо функцію як стратегію!
def process_data(data, sort_strategy):
    """strategy — звичайна функція або callable."""
    return sort_strategy(data)

# Передаємо вбудовані функції або лямбди як стратегії
print("Python:", process_data([3, 1, 4, 1, 5], sorted))
print("Python (зворотній):", process_data([3, 1, 4, 1, 5], lambda d: sorted(d, reverse=True)))

# А якщо стратегія потребує стану? Використовуємо callable object!
class WeightedSort:
    """Стратегія з пам'яттю: пріоритизує певні елементи."""
    def __init__(self, priority_item):
        self.priority_item = priority_item
        self.call_count = 0

    def __call__(self, data):
        self.call_count += 1
        # Пріоритетний елемент завжди перший
        return [self.priority_item] + [x for x in sorted(data) if x != self.priority_item]

priority_sort = WeightedSort(priority_item=4)
print("З пріоритетом:", process_data([3, 1, 4, 1, 5], priority_sort))
print("З пріоритетом:", process_data([9, 4, 2, 7], priority_sort))
print(f"Стратегію викликано: {priority_sort.call_count} рази")

**Висновок по Strategy:**

- Якщо стратегія **без стану** → передайте звичайну функцію
- Якщо стратегія **зі станом** → callable object (`__call__`)
- Повний клас із наслідуванням лише тоді, коли алгоритм складний і має багато методів

### Патерн «Команда» (Command Pattern)

**Класичне визначення:**  
Інкапсулює запит як об'єкт, дозволяючи підтримувати скасування, логування та чергу запитів.

**Класичний шаблон** вимагає:  
- Абстрактний `Command` із методом `execute()`  
- Окремий клас-команда для кожної дії  

**Python-спосіб:**  
Callable objects або замикання — залежно від потреби у стані.

In [ ]:
# Система команд з підтримкою скасування (undo) через __call__

class CommandHistory:
    """Журнал команд — підтримує скасування останньої дії."""
    def __init__(self):
        self._history = []

    def execute(self, command_fn):
        """Виконати команду і зберегти її в журналі."""
        result = command_fn()  # викликаємо об'єкт як функцію
        self._history.append(command_fn)
        return result

    def undo(self):
        """Скасувати останню команду."""
        if not self._history:
            print("Немає команд для скасування.")
            return
        last_command = self._history.pop()
        # Callable object може мати метод undo()
        if hasattr(last_command, 'undo'):
            last_command.undo()
        else:
            print("Ця команда не підтримує скасування.")


class AddTextCommand:
    """Команда: додати текст до документу."""
    def __init__(self, document: list, text: str):
        self.document = document
        self.text = text

    def __call__(self):
        # execute: додати текст
        self.document.append(self.text)
        print(f"Додано: '{self.text}'")
        return self

    def undo(self):
        # undo: видалити останній доданий текст
        if self.document and self.document[-1] == self.text:
            self.document.pop()
            print(f"Скасовано: '{self.text}' видалено")


# Демонстрація
doc = []
history = CommandHistory()

# Виконуємо команди
history.execute(AddTextCommand(doc, "Привіт, світ!"))
history.execute(AddTextCommand(doc, "Це Python."))
history.execute(AddTextCommand(doc, "ООП — потужно."))

print(f"\nДокумент: {doc}")

# Скасовуємо останню команду
print("\n--- Undo ---")
history.undo()
print(f"Документ після undo: {doc}")

**Ключовий висновок по патернах:**

| Патерн | Класичний ООП | Python-спосіб |
|--------|--------------|---------------|
| Strategy | Ієрархія класів із `execute()` | Передати функцію або callable |
| Command | Клас-команда з `execute()` і `undo()` | Callable object із `__call__` і `undo()` |
| Observer | Список слухачів з `update()` | Список функцій / callable |

> Python не скасовує патерни — він **спрощує їх реалізацію** завдяки функціям-першокласним об'єктам та duck typing.

---

## Підсумок уроку

| # | Концепція | Ключова ідея | Де використовувати |
|---|-----------|--------------|--------------------|
| 1 | **Поліморфізм** | Один виклик → різна поведінка залежно від об'єкта | Замість `if isinstance(...)` перевірок |
| 2 | **Duck Typing** | Python перевіряє поведінку (методи), а не тип | Коли не потрібна жорстка ієрархія |
| 3 | **Інкапсуляція** | Об'єкт захищає свої інваріанти від зовнішнього хаосу | Будь-які атрибути з бізнес-правилами |
| 4 | **`@property`** | Getter/setter без синтаксичного шуму | Валідація при присвоєнні атрибута |
| 5 | **`__str__`/`__repr__`** | Два різних рядкових представлення для двох аудиторій | Будь-який клас, де потрібен print |
| 6 | **`__add__`** | Перевантаження оператора `+` → новий об'єкт | Числові/векторні типи |
| 7 | **`__len__`** | Підключення до `len()` | Класи-контейнери |
| 8 | **`__call__`** | Об'єкт, що викликається як функція, але пам'ятає стан | Rate limiter, кеш, stateful callback |

---

### Ментальна модель: Рівні інтеграції з Python

```
Рівень 1: Базовий клас
  → Створюємо об'єкти, зберігаємо стан, визначаємо поведінку

Рівень 2: Дандер-методи
  → Наш клас «говорить» мовою Python Runtime
  → print(obj), len(obj), obj + other, obj() — всі працюють природно

Рівень 3: Протоколи (Duck Typing)
  → Наш клас вписується в будь-який код, що очікує певну поведінку
  → Без спадкування, без формальних інтерфейсів
```

---

### Що далі?

- **Урок 18:** Декоратори @property та Дескриптори.